# Replicate-Matched Integration Workflow

Use this notebook when biological replicates are **paired** across data types  
(e.g. TX and MX samples were collected from the same biological material at the same time points,  
and sample IDs match across datasets).

**Pipeline:**
```
linked_data
  → filter_data          (remove rare features)
  → devariance_data      (remove low-variance features)
  → replicate_filtering  (remove outlier replicates)
  → scale_data           (per-dataset: vst/zscore/etc. → brings data types to comparable scale)
  → integrate_data       (concatenate scaled matrices → features × samples)
  → normalize_integrated_data  (optional post-integration zscore)
  → feature_selection    (all methods valid: lfc, variance, glm, kruskal, etc.)
  → feature_grouping     (group features by abundance pattern similarity)
  → compare_groups_to_pathways
```

**Config files used:**
- `config_examples/data_processing.yml` — filtering, devariancing, replicate handling
- `config_examples/analysis_replicate_matched.yml` — scaling, post_integration, feature selection, feature grouping

# Project setup

In [ ]:
import os
import sys
import yaml
import pandas as pd
import tools.objects as objs
import tools.helpers as hlp
pd.options.display.max_columns = None
pd.set_option('display.max_colwidth', 500)

In [ ]:
# Optional: list all configurations available to load with the 'config_path' argument below
hlp.list_persistent_configs()

In [ ]:
# Initialize project (add the data processing and analysis hashes if resuming a previous project)
project = objs.Project(data_processing_hash=None, analysis_hash=None, overwrite=False)

# Dataset initiation

In [ ]:
# Create dataset objects and gather raw metadata and data (already pre-obtained from JGI sources)
tx_dataset = objs.TX(project)
mx_dataset = objs.MX(project, last=True)

# Analysis initiation

In [ ]:
# Create analysis object (collection of datasets and methods for performing integration)
# integration_mode is automatically detected as 'replicate_matched' from analysis_replicate_matched.yml
analysis = objs.Analysis(project, datasets=[tx_dataset, mx_dataset])
print(f"integration_mode: {analysis.integration_mode}")

# Link Samples Between Datasets

In [ ]:
# Link analysis datasets by finding corresponding sample metadata fields
analysis.link_metadata()

In [ ]:
# Link analysis datasets matrices by using linked metadata
analysis.link_data()

# Data processing

Steps shared by both branches. Parameters are set in `data_processing.yml`.

In [ ]:
# Filter out rare features from analysis datasets based on minimum observed value
analysis.filter_all_datasets()

In [ ]:
# Filter out features from analysis datasets that were not impacted by experimentation
analysis.devariance_all_datasets()

In [ ]:
# Filter out features with outlier replicates from each dataset
analysis.replicability_test_all_datasets()

In [ ]:
# View non-normalized data distributions before scaling
hlp.plot_simple_histogram(mx_dataset.replicate_filtered_data, plot_title="Metabolite Peak Heights (raw)", output_dir=analysis.output_dir)
hlp.plot_simple_histogram(tx_dataset.replicate_filtered_data, plot_title="Transcript Counts (raw)", output_dir=analysis.output_dir)

# Per-dataset scaling

Each dataset is scaled independently to bring different data types (RNA counts, metabolite peak heights)  
to a comparable scale before concatenation. Scaling method is set in `analysis_replicate_matched.yml → scaling`.

In [ ]:
# Scale each dataset independently (vst/zscore/etc.) → produces scaled_data per dataset
analysis.scale_all_datasets()

# Integration analysis

Scaled datasets are concatenated into a single features × samples matrix.  
An optional post-integration normalization pass can be applied (set in `analysis_replicate_matched.yml → post_integration`).

In [ ]:
# Integrate metadata tables by overlapping samples
analysis.integrate_metadata()

In [ ]:
# Integrate data matrices: concatenate scaled_data across datasets → integrated_data (features × samples)
analysis.integrate_data()

In [ ]:
# Optional post-integration normalization (e.g. zscore across all samples).
# Set post_integration.method: null in analysis_replicate_matched.yml to skip.
# Output: integrated_data_selected (features × samples)
analysis.normalize_integrated_data()

In [ ]:
# Annotate the integrated features with pre-generated feature annotation tables
analysis.annotate_integrated_features()

In [ ]:
# Subset features using statistical tests (all methods valid in replicate_matched mode)
analysis.perform_feature_selection()

In [ ]:
# View PCA after integration to examine clustering pattern of combined data types
hlp.plot_simple_pca(
    analysis.integrated_data_selected,
    analysis.integrated_metadata,
    title="PCA of replicate-matched integrated data",
    output_dir=analysis.output_dir
)

# Find data-driven groups of features

In [ ]:
# Group features by abundance pattern similarity across samples
analysis.group_features()

# Data-driven vs. knowledge-driven feature grouping

In [ ]:
# Compare data-driven feature groups against biological pathway annotations
grouping_results = analysis.compare_groups_to_pathways()

# Exploration of Results

In [ ]:
# Assess enrichment of an annotation layer in the extracted groups
#analysis.perform_functional_enrichment()

In [ ]:
# View average abundance of a given group across samples
analysis.plot_submodule_avg_abundance(submodule_name='group_1', metadata_cat='group', save_plot=True)

In [ ]:
# View abundance patterns of individual features
analysis.plot_individual_feature(feature_id='mx_11781_positive', metadata_cat='group', save_plot=True)

In [ ]:
# View pairwise feature correlation
hlp.plot_feature_pair_correlation(
    data=analysis.integrated_data_selected,
    metadata=analysis.integrated_metadata,
    feature_1="tx_gene_12834",
    feature_2="mx_11781_positive",
    color_by="timepoint",
    output_dir=analysis.output_dir
)

In [ ]:
# Sync all results tables to database for query actions
analysis.register_all_existing_data()

# Save configuration and notebook

In [ ]:
# Save persistent configuration and notebook files
project.save_persistent_config_and_notebook()